# 🏛️ ADOIT Data Product Pipeline## Domain: Enterprise Architecture | Catalog: `adoit_product` | Owner: EA TeamComplete **Bronze → Silver → Gold** pipeline for the Application Landscape data product.**Pipeline Flow:**```ADOIT CSVs → Bronze (raw_applications, raw_capabilities, raw_map)                ↓             Silver (applications, business_capabilities) — enriched                ↓             Gold (application_landscape) — joined 360° view```

## 🪨 Bronze Layer — Raw Ingestion from ADOITRaw data as extracted from ADOIT REST API. No transformations.

In [ ]:
# Load CSV data into bronze tablesimport os, pandas as pdnotebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()bundle_root = notebook_path.rsplit('/src/notebooks/', 1)[0]data_dir = f"/Workspace{bundle_root}/src/data"if not os.path.exists(data_dir):    data_dir = "/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/dbx-dps-raise/src/data"print(f"Data directory: {data_dir}")for csv_file, table_name in [    ("adoit_applications.csv", "adoit_product.bronze.raw_applications"),    ("adoit_capabilities.csv", "adoit_product.bronze.raw_business_capabilities"),    ("adoit_app_capability_map.csv", "adoit_product.bronze.raw_app_capability_map")]:    csv_path = f"{data_dir}/{csv_file}"    pdf = pd.read_csv(csv_path)    df = spark.createDataFrame(pdf)    df.write.mode("overwrite").saveAsTable(table_name)    print(f"Loaded {len(pdf)} rows -> {table_name}")

In [ ]:
%sql-- Verify bronze: ApplicationsSELECT * FROM adoit_product.bronze.raw_applications ORDER BY app_id

In [ ]:
%sql-- Verify bronze: Business capabilitiesSELECT * FROM adoit_product.bronze.raw_business_capabilities ORDER BY capability_id

## ⚙️ Silver Layer — Curated & Quality-ScoredTransformations: standardization, enrichment (age, cloud flag), quality scoring.

In [ ]:
%sqlMERGE INTO adoit_product.silver.applications AS targetUSING (    SELECT         app_id, TRIM(app_name) AS app_name, app_owner,        INITCAP(department) AS department, lifecycle_status, criticality, technology_stack,        go_live_date,        ROUND(DATEDIFF(CURRENT_DATE(), go_live_date) / 365.25, 1) AS app_age_years,        CASE             WHEN LOWER(technology_stack) LIKE '%cloud%' OR LOWER(technology_stack) LIKE '%saas%'                 OR LOWER(technology_stack) LIKE '%azure%' OR LOWER(technology_stack) LIKE '%databricks%' THEN true            ELSE false        END AS is_cloud_hosted,        ROUND((CASE WHEN app_owner IS NOT NULL THEN 0.25 ELSE 0 END +               CASE WHEN department IS NOT NULL THEN 0.25 ELSE 0 END +               CASE WHEN lifecycle_status IS NOT NULL THEN 0.25 ELSE 0 END +               CASE WHEN criticality IS NOT NULL THEN 0.25 ELSE 0 END), 2) AS quality_score,        current_timestamp() AS processed_at    FROM adoit_product.bronze.raw_applications    WHERE app_id IS NOT NULL) AS sourceON target.app_id = source.app_idWHEN MATCHED THEN UPDATE SET *WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sqlMERGE INTO adoit_product.silver.business_capabilities AS targetUSING (    SELECT capability_id, capability_name, capability_domain,           business_value, maturity_level, current_timestamp() AS ingested_at    FROM adoit_product.bronze.raw_business_capabilities) AS sourceON target.capability_id = source.capability_idWHEN MATCHED THEN UPDATE SET *WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sqlSELECT app_id, app_name, app_owner, department, lifecycle_status, criticality,       is_cloud_hosted, app_age_years, quality_scoreFROM adoit_product.silver.applications ORDER BY quality_score DESC

## 🏆 Gold Layer — Application Landscape Data ProductJoins applications with business capabilities for a 360° portfolio view.

In [ ]:
%sqlMERGE INTO adoit_product.gold.application_landscape AS targetUSING (    SELECT         a.app_id, a.app_name, a.app_owner, a.department,        a.lifecycle_status, a.criticality, a.technology_stack,        a.is_cloud_hosted, a.app_age_years,        COUNT(DISTINCT m.capability_id) AS capability_count,        CONCAT_WS(', ', COLLECT_SET(CASE WHEN m.support_level = 'Primary' THEN c.capability_name END)) AS primary_capabilities,        CONCAT_WS(', ', COLLECT_SET(c.capability_name)) AS all_capabilities,        a.quality_score,        current_timestamp() AS product_generated_at    FROM adoit_product.silver.applications a    LEFT JOIN adoit_product.bronze.raw_app_capability_map m ON a.app_id = m.app_id    LEFT JOIN adoit_product.silver.business_capabilities c ON m.capability_id = c.capability_id    GROUP BY a.app_id, a.app_name, a.app_owner, a.department,             a.lifecycle_status, a.criticality, a.technology_stack,             a.is_cloud_hosted, a.app_age_years, a.quality_score) AS sourceON target.app_id = source.app_idWHEN MATCHED THEN UPDATE SET *WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sqlSELECT app_id, app_name, app_owner, department, criticality, lifecycle_status,       is_cloud_hosted, ROUND(app_age_years, 1) AS age_yrs, capability_count,       primary_capabilities, quality_scoreFROM adoit_product.gold.application_landscapeORDER BY criticality, app_name

## ✅ Pipeline Complete- **Bronze**: 10 applications + 10 capabilities + 13 mappings- **Silver**: 10 applications (enriched) + 10 capabilities- **Gold**: 10 application landscape records (joined 360° view)Next: Run `02_Adoit_Governance` for all data product characteristics.